<div style="background:linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%);color:#fff;padding:26px 30px;border-radius:14px">
<div style="font-size:30px;font-weight:700;line-height:1.2">Best public QCC portfolio: 392k-Toffoli DQI + 62-qubit invsqrt splice</div>
<div style="font-size:15px;margin-top:10px;opacity:.92">Google Verifiable Quantum Circuit Compilation &middot; every file re-verified with the unchanged official verifiers &middot; full credits below</div>
<div style="margin-top:14px"><span style="background:#ffd166;color:#0b1f3a;padding:4px 10px;border-radius:999px;font-weight:700">GM 25.4495 (local, default salt)</span>
<span style="background:rgba(255,255,255,.18);padding:4px 10px;border-radius:999px;margin-left:6px">DQI 0 failures at 3 salts</span>
<span style="background:rgba(255,255,255,.18);padding:4px 10px;border-radius:999px;margin-left:6px">uses the QCC Circuit Atlas</span></div>
</div>

This notebook joins the best public file of each task, as of 2026-11-12 17:00 UTC, and re-verifies all three with the
official verifiers. **The new combination:** maoxin123's 392k-Toffoli DQI decoder `4c81b726` (published today, about
2,070 Toffolis below the previous best decoder) had not yet been combined with the best invsqrt (our splice `9c708f7b`)
and the best chemistry (maoxin123's 4,089-Toffoli loaders with Heliosli's weights). Together they score about
**+0.04 over the previous public best**. Almost all of it is other people's work: every file and every improvement is
credited below, with the author, the change and a link. If this notebook helps you, an upvote is appreciated.

| Task | File | Qubits | Built by (latest change first) |
|---|---|---:|---|
| invsqrt | `9c708f7b` | 62 | **our splice** of WCao24's 44-CSWAP normaliser (`94b29cc5`) and Thomas Gamet's early prefix (`5ec4f516`), on maoxin123's and manwithacat's 62-qubit line |
| DQI | `5778cdd5` | 892 | **maoxin123's** 392k-Toffoli rewrite `4c81b726` of `cd71c482` + 2 retained products (Areeb Ali's method, our planner) |
| chemistry | npz `e9cffb84` + 15 loaders | 427 | **maoxin123's** 4,089-Toffoli loaders; Heliosli's weight update of prvsiyan's refit; Lera (marculera)'s (8, 143, 44) table |

## Acknowledgements
Thank you to everyone whose public work this notebook stands on. In particular **maoxin123** for the DQI decoder and
the chemistry loaders used here (and the invsqrt base before that); **Heliosli** for the weight update and the
*qcc-circuits-live* dataset; **Areeb Ali** for product retention, the previous public best; **Alex Chilton**, whose
*Verified Three-Task Portfolio* set the standard for pinning and re-verifying the frontier; **Atakan Aldemir**,
**WCao24**, **Thomas Gamet**, **prvsiyan**, **manwithacat (James Barlow)**, **Ertuğrul Özer (ozertuu)**, **Lera
(marculera)**, **Kirill Tymchishin**, **Syed Asad Ali**, **SomeJoint**, **busyaprime (Akmal Xodarev)**, **yoyobar**,
**mrmeows**, **Hosen42**, **Gengsr**, **Ahmed Berat Özer**, **Isabe11aM00re**, **Ali Haydar Özdağ**, the team
**i want to run gemma 9b**, **jday96314** and **Oleg Sher (sheroleg)**. And Google Quantum AI and Kaggle for the
competition, the open kickmix simulator and the verifiers. If we missed you or misattributed a change, please tell us
in the comments and we will fix it in the next version.

**What is ours here:** two more retained DQI products on `4c81b726` (our extension of Areeb Ali's product retention,
a small gain of 1.6 Toffolis per shot, section below), the invsqrt splice `9c708f7b` (1,213.87 Toffolis per shot against 1,216.83 for the best
circuit before it), this combination and its verification, and the
[QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) dataset this notebook reads
for the provenance table (every public circuit, deduplicated, with its first source and author).
Companion notebook: [QCC Toolkit: Simulator & Cost Anatomy](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).

<div style="background:#fff7e0;border-left:6px solid #ffb703;padding:12px 16px;border-radius:8px;color:#1b1b1b">
<b>Getting the submission file.</b> This public version verifies everything and prints the scores, but does not leave
<code>submission.zip</code> in its output. To get it: <b>Copy &amp; Edit</b> this notebook, set
<code>WRITE_SUBMISSION = True</code> in the next cell, and <b>Run All</b> (about 25 minutes on CPU); the file appears in
<code>/kaggle/working</code>. If the notebook helps you, an upvote keeps it maintained, and please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = False
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, shutil, math, lzma, csv
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
import kickmix as km
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
sha = lambda b: hashlib.sha256(b).hexdigest()
def members(zpath):
    with zipfile.ZipFile(zpath) as z: return {n: z.read(n) for n in z.namelist()}
zips = {str(p): p for p in inp.rglob("submission.zip")}
wcao = members(next(p for k, p in zips.items() if "deterministic-open-portfolio" in k))
thom = members(next(p for k, p in zips.items() if "new-base-bundle" in k))
inv = wcao["invsqrt.kmb"]; assert sha(inv).startswith("94b29cc5"), sha(inv)
# DQI: maoxin123's 392k-Toffoli decoder (dataset maoxin123/qcc-dqi-392k-toffoli-decoder, CC0)
DQI_SHA = "4c81b726"
dqi = None
for xz in inp.rglob("dqi.kmb.xz"):
    b = lzma.decompress(xz.read_bytes())
    if sha(b).startswith(DQI_SHA): dqi = b; log("dqi from", xz); break
assert dqi is not None, "4c81b726 not found"
# chemistry: Heliosli's qcc-circuits-live dataset (maoxin123's 4,089-Toffoli loaders + Heliosli's weight update)
HEL = next(p.parent for p in inp.rglob("QCC_RELEASE_MANIFEST.json"))
man = json.load(open(HEL / "QCC_RELEASE_MANIFEST.json"))
chem = {f["file"]: (HEL / f["stored_as"]).read_bytes() for f in man["files"] if f["file"].startswith("dfthcsos/")}
for f in man["files"]:
    if f["file"] in chem: assert sha(chem[f["file"]]) == f["sha256"], f
npz = chem["dfthcsos/submission.npz"]; assert sha(npz).startswith("e9cffb84"), sha(npz)
assert sha(chem["dfthcsos/angle_k_0.kmb"]).startswith("77f5f6d8")
log("dqi", sha(dqi)[:12], "chem npz", sha(npz)[:12])
(SUB / "dfthcsos" / "submission.npz").write_bytes(npz)
for n, b in chem.items():
    if n.startswith("dfthcsos/angle_k_"): (SUB / n).write_bytes(b)
(SUB / "dqi.kmb").write_bytes(dqi)

## DQI: two more retained products (our extension of Areeb Ali's product retention)
Areeb Ali's method keeps a repeated AND product in a scratch qubit that is idle between its two occurrences, so the
second Toffoli becomes two `CX` gates and a free measurement-based uncompute. On maoxin123's `4c81b726` the 52 scratch
qubits are almost never idle (all 52 are busy for 5.4M of its 57M lines), so his planner finds only 2 of 3,525
repeated pairs. Our extension renames scratch lifetimes between qubits (interval colouring), so a product needs one
free scratch qubit at every line of its span instead of one qubit idle for the whole span; on this decoder it
retains the same 2 pairs (the limit is capacity, not fragmentation). Saving: 2 static Toffolis, 1.6 per shot,
verified with 0 failures at the default salt, `s1` and `s2`. The code below is Areeb Ali's scan and rewrite
(Apache 2.0) with our planner; every rewrite class was checked exhaustively on 1,497 random small circuits (all
inputs, all measurement outcomes). The build is deterministic and pinned by SHA-256; if the pin ever failed, the
notebook would fall back to `4c81b726` unchanged.

In [ ]:
'''retain2: product retention with scratch-lifetime renaming.
Derived from Areeb Ali's `retain.py` (notebook product-retention-133-free-toffolis-lb-25-40767, Apache 2.0): the value-fingerprint
scan, idle windows, rewrite patterns and the HMR/CZ uncompute are his. New here: busy periods + global interval colouring.'''
import random, collections, bisect, re
import numpy as np

def read_registers(path):
    import subprocess
    regq = set()
    out = subprocess.run(['grep', '-a', '^APPEND_TO_REGISTER', path], capture_output=True, text=True).stdout
    for line in out.splitlines():
        p = line.split()
        if p[1][0] == 'q': regq.add(p[1])
    return regq

def scan(path, regq, nq, seed=20261112):
    '''Forward pass with value fingerprints (Areeb). Returns occ, idle, n, maxbit.
    occ: product fp -> [(line,a,b,t)]; idle: scratch qubit -> [(z, L, ins)] windows (zero & untouched after line z, before L).
    Scratch qubits start idle with z=-1 (new vs Areeb).'''
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    memo = {}; occ = collections.defaultdict(list); idle = collections.defaultdict(list)
    zs = {}
    for i in range(nq):
        q = f"q{i}"
        if q not in regq: zs[q] = -1
    depth = 0; written = []; block_start = [0]; maxbit = 0
    bre = re.compile(r'b(\d+)')
    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0; free = True
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
            free = False
        return core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0), free
    def touch(q, L):
        z = zs.pop(q, None)
        if z is not None:
            ins = (L - 1) if depth == 0 else (block_start[0] - 1)
            if ins >= z: idle[q].append((z, L, ins))
    def settle(q, L):
        if depth == 0 and h[q] == 0 and q not in regq: zs[q] = L
        if written: written[-1].add(q)
    lineno = -1
    with open(path) as f:
        for lineno, line in enumerate(f):
            c0 = line[0]
            if c0 == 'C' and line[1] == 'X':
                p = line.split()
                if len(p) == 3:
                    a = p[1]; b = p[2]
                    if a in zs: touch(a, lineno); zs[a] = lineno
                    if b in zs: touch(b, lineno)
                    v = h[b] ^ h[a]; h[b] = v
                    if depth == 0:
                        if not v and b not in regq: zs[b] = lineno
                    else: written[-1].add(b)
                    continue
            p = line.split()
            if not p: continue
            op = p[0]
            if op == 'BIT_INVERT' or op == 'BIT_STORE0' or op == 'BIT_STORE1':
                for t in p[1:]:
                    if t[0] == 'b':
                        v = int(t[1:]); 
                        if v > maxbit: maxbit = v
                continue
            if op in ('INFO_TAG', 'APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT', 'NEG'): continue
            if op == 'PUSH_CONDITION':
                if depth == 0: block_start[0] = lineno
                depth += 1; written.append(set()); continue
            if op == 'POP_CONDITION':
                if depth:
                    depth -= 1; w = written.pop()
                    for q in w: h[q] = rnd(64); zs.pop(q, None)
                    if written: written[-1] |= w
                continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            for t in p[1:]:
                if t[0] == 'b':
                    v = int(t[1:])
                    if v > maxbit: maxbit = v
            if cond is not None:
                v = int(cond[1:])
                if v > maxbit: maxbit = v
            qs = [t for t in p[1:] if t[0] == 'q']
            zero_before = {q: (q in zs) for q in qs}
            for q in qs: touch(q, lineno)
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                if cond is None and depth == 0:
                    for q in qs:
                        if zero_before[q]: zs[q] = lineno
                continue
            if cond is not None:
                for q in (qs[-1:] if op in ('X', 'CX', 'CCX', 'HMR', 'R') else qs if op in ('SWAP',) else qs[1:]):
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
                for q in qs:
                    if zero_before[q] and h[q] == 0 and depth == 0: zs[q] = lineno
                continue
            if op == 'X': t = qs[0]; h[t] ^= ONE; settle(t, lineno)
            elif op == 'CX':
                h[qs[1]] ^= h[qs[0]]; settle(qs[1], lineno); settle(qs[0], lineno)
            elif op == 'SWAP':
                a, b = qs; h[a], h[b] = h[b], h[a]; settle(a, lineno); settle(b, lineno)
            elif op in ('R', 'HMR'):
                t = qs[0]; h[t] = 0; settle(t, lineno)
            elif op == 'CCX':
                a, b, t = qs
                P, free = AND(h[a], h[b])
                if not free and P and depth == 0: occ[P].append((lineno, a, b, t))
                h[t] ^= P; settle(t, lineno); settle(a, lineno); settle(b, lineno)
            elif op == 'CSWAP':
                c, a, b = qs
                P, free = AND(h[c], h[a] ^ h[b]); h[a] ^= P; h[b] ^= P
                for q in qs: settle(q, lineno)
            else:
                for q in qs:
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
    n = lineno + 1
    for q, z in zs.items():
        if n - 1 >= z: idle[q].append((z, n, n - 1))
    return occ, idle, n, maxbit


def scan_snap(path, regq, nq, snap_lines, snaps, lmsnap, seed=20261112):
    '''Forward pass with value fingerprints (Areeb). Returns occ, idle, n, maxbit.
    occ: product fp -> [(line,a,b,t)]; idle: scratch qubit -> [(z, L, ins)] windows (zero & untouched after line z, before L).
    Scratch qubits start idle with z=-1 (new vs Areeb).'''
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    memo = {}; occ = collections.defaultdict(list); idle = collections.defaultdict(list)
    zs = {}
    for i in range(nq):
        q = f"q{i}"
        if q not in regq: zs[q] = -1
    depth = 0; written = []; block_start = [0]; maxbit = 0
    lm = {}
    C64 = 0x9E3779B97F4A7C15
    qn = [f'q{i}' for i in range(nq)]
    bre = re.compile(r'b(\d+)')
    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0; free = True
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
            free = False
        return core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0), free
    def touch(q, L):
        z = zs.pop(q, None)
        if z is not None:
            ins = (L - 1) if depth == 0 else (block_start[0] - 1)
            if ins >= z: idle[q].append((z, L, ins))
    def settle(q, L):
        if depth == 0 and h[q] == 0 and q not in regq: zs[q] = L
        if written: written[-1].add(q)
    lineno = -1
    with open(path) as f:
        for lineno, line in enumerate(f):
            if lineno in snap_lines:
                snaps[lineno] = np.array([(h[q] & LOW) ^ (C64 if (h[q] >> 64) else 0) for q in qn], dtype=np.uint64)
                lmsnap[lineno] = np.array([lm.get(q, -1) for q in qn], dtype=np.int64)
            c0 = line[0]
            if c0 == 'C' and line[1] == 'X':
                p = line.split()
                if len(p) == 3:
                    a = p[1]; b = p[2]; lm[a] = lineno; lm[b] = lineno
                    if a in zs: touch(a, lineno); zs[a] = lineno
                    if b in zs: touch(b, lineno)
                    v = h[b] ^ h[a]; h[b] = v
                    if depth == 0:
                        if not v and b not in regq: zs[b] = lineno
                    else: written[-1].add(b)
                    continue
            p = line.split()
            if not p: continue
            op = p[0]
            if op == 'BIT_INVERT' or op == 'BIT_STORE0' or op == 'BIT_STORE1':
                for t in p[1:]:
                    if t[0] == 'b':
                        v = int(t[1:]); 
                        if v > maxbit: maxbit = v
                continue
            if op in ('INFO_TAG', 'APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT', 'NEG'): continue
            if op == 'PUSH_CONDITION':
                if depth == 0: block_start[0] = lineno
                depth += 1; written.append(set()); continue
            if op == 'POP_CONDITION':
                if depth:
                    depth -= 1; w = written.pop()
                    for q in w: h[q] = rnd(64); zs.pop(q, None)
                    if written: written[-1] |= w
                continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            for t in p[1:]:
                if t[0] == 'b':
                    v = int(t[1:])
                    if v > maxbit: maxbit = v
            if cond is not None:
                v = int(cond[1:])
                if v > maxbit: maxbit = v
            qs = [t for t in p[1:] if t[0] == 'q']
            for q in qs: lm[q] = lineno
            zero_before = {q: (q in zs) for q in qs}
            for q in qs: touch(q, lineno)
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                if cond is None and depth == 0:
                    for q in qs:
                        if zero_before[q]: zs[q] = lineno
                continue
            if cond is not None:
                for q in (qs[-1:] if op in ('X', 'CX', 'CCX', 'HMR', 'R') else qs if op in ('SWAP',) else qs[1:]):
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
                for q in qs:
                    if zero_before[q] and h[q] == 0 and depth == 0: zs[q] = lineno
                continue
            if op == 'X': t = qs[0]; h[t] ^= ONE; settle(t, lineno)
            elif op == 'CX':
                h[qs[1]] ^= h[qs[0]]; settle(qs[1], lineno); settle(qs[0], lineno)
            elif op == 'SWAP':
                a, b = qs; h[a], h[b] = h[b], h[a]; settle(a, lineno); settle(b, lineno)
            elif op in ('R', 'HMR'):
                t = qs[0]; h[t] = 0; settle(t, lineno)
            elif op == 'CCX':
                a, b, t = qs
                P, free = AND(h[a], h[b])
                if not free and P and depth == 0: occ[P].append((lineno, a, b, t))
                h[t] ^= P; settle(t, lineno); settle(a, lineno); settle(b, lineno)
            elif op == 'CSWAP':
                c, a, b = qs
                P, free = AND(h[c], h[a] ^ h[b]); h[a] ^= P; h[b] ^= P
                for q in qs: settle(q, lineno)
            else:
                for q in qs:
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
    n = lineno + 1
    for q, z in zs.items():
        if n - 1 >= z: idle[q].append((z, n, n - 1))
    return occ, idle, n, maxbit



def scan_span(path, regq, nq, hits, stats, seed=20261112):
    '''Forward pass with value fingerprints (Areeb). Returns occ, idle, n, maxbit.
    occ: product fp -> [(line,a,b,t)]; idle: scratch qubit -> [(z, L, ins)] windows (zero & untouched after line z, before L).
    Scratch qubits start idle with z=-1 (new vs Areeb).'''
    rnd = random.Random(seed).getrandbits
    ONE = 1 << 64; LOW = ONE - 1
    h = collections.defaultdict(int)
    for q in regq: h[q] = rnd(64)
    memo = {}; occ = collections.defaultdict(list); idle = collections.defaultdict(list)
    zs = {}
    for i in range(nq):
        q = f"q{i}"
        if q not in regq: zs[q] = -1
    depth = 0; written = []; block_start = [0]; maxbit = 0
    lm = {}
    C64 = 0x9E3779B97F4A7C15
    qn = [f'q{i}' for i in range(nq)]
    bre = re.compile(r'b(\d+)')
    def AND(x, y):
        xn, x0, yn, y0 = x >> 64, x & LOW, y >> 64, y & LOW
        if x0 == 0 or y0 == 0 or x0 == y0:
            core = x0 if (x0 == y0) else 0; free = True
        else:
            k = (x0, y0) if x0 < y0 else (y0, x0)
            core = memo.get(k)
            if core is None: core = memo[k] = rnd(64)
            free = False
        return core ^ (y0 if xn else 0) ^ (x0 if yn else 0) ^ (ONE if (xn and yn) else 0), free
    def touch(q, L):
        z = zs.pop(q, None)
        if z is not None:
            ins = (L - 1) if depth == 0 else (block_start[0] - 1)
            if ins >= z: idle[q].append((z, L, ins))
    def settle(q, L):
        if depth == 0 and h[q] == 0 and q not in regq: zs[q] = L
        if written: written[-1].add(q)
    lineno = -1
    with open(path) as f:
        for lineno, line in enumerate(f):
            c0 = line[0]
            if c0 == 'C' and line[1] == 'X':
                p = line.split()
                if len(p) == 3:
                    a = p[1]; b = p[2]; lm[a] = lineno; lm[b] = lineno
                    if a in zs: touch(a, lineno); zs[a] = lineno
                    if b in zs: touch(b, lineno)
                    v = h[b] ^ h[a]; h[b] = v
                    if depth == 0:
                        if not v and b not in regq: zs[b] = lineno
                    else: written[-1].add(b)
                    continue
            p = line.split()
            if not p: continue
            op = p[0]
            if op == 'BIT_INVERT' or op == 'BIT_STORE0' or op == 'BIT_STORE1':
                for t in p[1:]:
                    if t[0] == 'b':
                        v = int(t[1:]); 
                        if v > maxbit: maxbit = v
                continue
            if op in ('INFO_TAG', 'APPEND_TO_REGISTER', 'REGISTER', 'DEBUG_PRINT', 'NEG'): continue
            if op == 'PUSH_CONDITION':
                if depth == 0: block_start[0] = lineno
                depth += 1; written.append(set()); continue
            if op == 'POP_CONDITION':
                if depth:
                    depth -= 1; w = written.pop()
                    for q in w: h[q] = rnd(64); zs.pop(q, None)
                    if written: written[-1] |= w
                continue
            cond = None
            if len(p) >= 3 and p[-2] == 'if': cond = p[-1]; p = p[:-2]
            for t in p[1:]:
                if t[0] == 'b':
                    v = int(t[1:])
                    if v > maxbit: maxbit = v
            if cond is not None:
                v = int(cond[1:])
                if v > maxbit: maxbit = v
            qs = [t for t in p[1:] if t[0] == 'q']
            for q in qs: lm[q] = lineno
            zero_before = {q: (q in zs) for q in qs}
            for q in qs: touch(q, lineno)
            if op in ('Z', 'CZ', 'CCZ', 'Z_POW'):
                if cond is None and depth == 0:
                    for q in qs:
                        if zero_before[q]: zs[q] = lineno
                continue
            if cond is not None:
                for q in (qs[-1:] if op in ('X', 'CX', 'CCX', 'HMR', 'R') else qs if op in ('SWAP',) else qs[1:]):
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
                for q in qs:
                    if zero_before[q] and h[q] == 0 and depth == 0: zs[q] = lineno
                continue
            if op == 'X': t = qs[0]; h[t] ^= ONE; settle(t, lineno)
            elif op == 'CX':
                h[qs[1]] ^= h[qs[0]]; settle(qs[1], lineno); settle(qs[0], lineno)
            elif op == 'SWAP':
                a, b = qs; h[a], h[b] = h[b], h[a]; settle(a, lineno); settle(b, lineno)
            elif op in ('R', 'HMR'):
                t = qs[0]; h[t] = 0; settle(t, lineno)
            elif op == 'CCX':
                a, b, t = qs
                P, free = AND(h[a], h[b])
                if not free and P and depth == 0:
                    occ[P].append((lineno, a, b, t))
                    stats['ccx'] += 1
                    vals = np.array([(h[q] & LOW) ^ (C64 if (h[q] >> 64) else 0) for q in qn], dtype=np.uint64)
                    ti = int(t[1:]); vals[ti] = 0
                    Pc = np.uint64((P & LOW) ^ (C64 if (P >> 64) else 0))
                    one = np.nonzero(vals == Pc)[0]
                    if len(one): hits.append((lineno, 1, tuple(int(x) for x in one[:3]), a, b, t)); stats['one'] += 1
                    else:
                        xs = vals ^ Pc
                        m = np.nonzero(np.isin(xs, vals) & (vals != 0))[0]
                        if len(m): hits.append((lineno, 2, tuple(int(x) for x in m[:3]), a, b, t)); stats['two'] += 1
                h[t] ^= P; settle(t, lineno); settle(a, lineno); settle(b, lineno)
            elif op == 'CSWAP':
                c, a, b = qs
                P, free = AND(h[c], h[a] ^ h[b]); h[a] ^= P; h[b] ^= P
                for q in qs: settle(q, lineno)
            else:
                for q in qs:
                    h[q] = rnd(64)
                    if written: written[-1].add(q)
    n = lineno + 1
    for q, z in zs.items():
        if n - 1 >= z: idle[q].append((z, n, n - 1))
    return occ, idle, n, maxbit



def busy_periods(idle, n):
    '''per scratch qubit: list of closed line ranges (S, Z) in which the qubit is not provably zero-and-untouched.
    Windows (z,L,ins): idle after line z until L; busy period i = [ins_i+1, z_{i+1}] (last: to n-1 if the final window does not reach n).'''
    per = {}; tails = {}
    for q, ws in idle.items():
        ws = sorted(ws); out = []
        for i in range(len(ws) - 1):
            S = ws[i][2] + 1; Z = ws[i + 1][0]
            assert S <= Z and (not out or out[-1][1] < S), (q, ws[i], ws[i+1])
            out.append((S, Z))
        last = ws[-1]
        if last[1] < n:     # last window ends before the file does: busy until the end
            out.append((last[2] + 1, n - 1)); tails[q] = (last[2] + 1, n - 1)
        per[q] = out
    busy_periods.tails = tails
    return per

def plan(occ, per, nscratch, first_new_bit, order='span', max_span=None, forbid=()):
    '''Choose repeated-product pairs to retain so that, at every line, (busy scratch lifetimes + retained products) <= nscratch,
    then colour all lifetimes onto the scratch qubits (interval colouring), renaming whole busy periods.
    Returns (runs, colour, stats). runs: list of (P, k0, k1) = occurrences k0..k1 of product P held in one scratch qubit.
    colour: dict (q,S,Z)->new qubit for busy periods ; run_host: run index -> qubit.'''
    pairs = []
    for P, lst in occ.items():
        for k in range(len(lst) - 1):
            st, en = lst[k][0], lst[k + 1][0]
            if max_span is None or en - st <= max_span: pairs.append((st, en, P, k))
    pts = set()
    for q, ps in per.items():
        for S, Z in ps: pts.add(S); pts.add(Z + 1)
    for st, en, P, k in pairs: pts.update((st, st + 1, en, en + 1))
    B = np.array(sorted(pts), dtype=np.int64); m = len(B)
    diff = np.zeros(m + 1, dtype=np.int32)
    for q, ps in per.items():
        for S, Z in ps:
            diff[np.searchsorted(B, S)] += 1; diff[np.searchsorted(B, Z + 1)] -= 1
    cov = np.cumsum(diff)[:m].astype(np.int32)
    base_max = int(cov.max())
    if order == 'span': pairs.sort(key=lambda x: (x[1] - x[0], x[0]))
    elif order == 'end': pairs.sort(key=lambda x: (x[1], x[0]))
    elif order == 'spanseg':
        pairs.sort(key=lambda x: (np.searchsorted(B, x[1]) - np.searchsorted(B, x[0]), x[0]))
    chosen = set()
    for st, en, P, k in pairs:
        i0 = int(np.searchsorted(B, st)); i1 = int(np.searchsorted(B, en + 1))
        sub = cov[i0:i1] + 1
        if (P, k - 1) in chosen: sub[0] -= 1
        if (P, k + 1) in chosen: sub[-1] -= 1
        if sub.max() <= nscratch:
            cov[i0:i1] = sub; chosen.add((P, k))
    # merge consecutive chosen pairs into runs
    runs = []
    for P, lst in occ.items():
        k = 0; L = len(lst)
        while k < L - 1:
            if (P, k) in chosen:
                k1 = k
                while (P, k1 + 1) in chosen: k1 += 1
                runs.append((P, k, k1 + 1)); k = k1 + 1
            else: k += 1
    return runs, dict(pairs=len(pairs), chosen=len(chosen), runs=len(runs), base_max_busy=base_max, final_max=int(cov.max()))

def colour(occ, per, runs, scratch):
    tails = getattr(busy_periods, 'tails', {})
    pin = {q: v[0] for q, v in tails.items()}
    '''Interval colouring of busy periods + run intervals on the scratch qubits. Prefers each period's original qubit.'''
    scratch = sorted(scratch)
    items = []                                   # (start, end, kind, ref)
    for q, ps in per.items():
        for S, Z in ps: items.append((S, Z, 0, (q, S, Z)))
    for ri, (P, k0, k1) in enumerate(runs):
        lst = occ[P]; items.append((lst[k0][0], lst[k1][0], 1, ri))
    items.sort(key=lambda x: (x[0], x[2]))
    nxt = {q: sorted(S for S, Z in per.get(q, [])) for q in scratch}
    import heapq
    free = set(scratch); active = []             # heap (end, qubit)
    pc = {}; rh = {}
    tailset = {(q, v[0], v[1]) for q, v in tails.items()}
    for S, E, kind, ref in items:
        while active and active[0][0] < S:
            e, q = heapq.heappop(active); free.add(q)
        if kind == 0:
            q = ref[0]
            if ref in tailset: q = ref[0]; assert q in free, 'pinned tail conflict'
            elif q not in free or (q in pin and E >= pin[q]):
                cs = [c for c in free if not (c in pin and E >= pin[c])]
                q = max(cs, key=lambda c: (bisect.bisect_left(nxt[c], S) >= len(nxt[c]) and 1 << 60 or nxt[c][bisect.bisect_left(nxt[c], S)], c))
            pc[ref] = q
        else:
            cs = [c for c in free if not (c in pin and E >= pin[c])]
            q = max(cs, key=lambda c: (bisect.bisect_left(nxt[c], S) >= len(nxt[c]) and 1 << 60 or nxt[c][bisect.bisect_left(nxt[c], S)], c))
            rh[ref] = q
        free.discard(q); heapq.heappush(active, (E, q))
    return pc, rh

def rewrite(src, dst, occ, per, runs, pc, rh, first_new_bit):
    '''Stream src to dst: rename busy periods per pc, replace occurrences of retained runs.'''
    import collections
    act = {}                                     # qubit -> new qubit (active renames)
    start_ev = collections.defaultdict(list); end_ev = collections.defaultdict(list)
    for (q, S, Z), q2 in pc.items():
        if q2 != q: start_ev[S].append((q, q2)); end_ev[Z + 1].append(q)
    tmpl = {}; nb = first_new_bit
    for ri, (P, k0, k1) in enumerate(runs):
        lst = occ[P]; s = rh[ri]
        for k in range(k0, k1 + 1):
            L, a, b, t = lst[k]
            tmpl[L] = (ri, k - k0, k1 - k0, nb if k == k1 else None)
        nb += 1
    nlines = 0
    with open(src) as f, open(dst, 'w') as g:
        for i, line in enumerate(f):
            if i in end_ev:
                for q in end_ev[i]: del act[q]
            if i in start_ev:
                for q, q2 in start_ev[i]: act[q] = q2
            if act or i in tmpl:
                p = line.split()
                if act and p:
                    for j in range(1, len(p)):
                        v = act.get(p[j])
                        if v is not None: p[j] = v
                if i in tmpl:
                    ri, pos, last, bit = tmpl[i]; s = rh[ri]
                    a, b, t = p[1], p[2], p[3]
                    out = []
                    if pos == 0: out.append(f"CCX {a} {b} {s}\n")
                    out.append(f"CX {s} {t}\n")
                    if pos == last: out += [f"HMR {s} b{bit}\n", f"CZ {a} {b} if b{bit}\n"]
                    g.writelines(out)
                else:
                    g.write(' '.join(p) + '\n' if (act and p) else line)
            else: g.write(line)
            nlines += 1
    return nlines, nb - first_new_bit

In [ ]:
RET_KMB = "5778cdd5d710567b970544b8929547cf3be37fe84f2a11643d49edc74eca96cc"
km.Circuit.from_file(SUB / "dqi.kmb", format="kmb").to_file("/tmp/dqi_base.kmx", format="kmx")
regq = read_registers("/tmp/dqi_base.kmx")
occ, idle, n, maxbit = scan("/tmp/dqi_base.kmx", regq, 892)
per = busy_periods(idle, n)
runs, st = plan(occ, per, 52, maxbit + 1, order="end"); log("retention plan", st)
pc, rh = colour(occ, per, runs, [f"q{i}" for i in range(840, 892)])
nl, nb = rewrite("/tmp/dqi_base.kmx", "/tmp/dqi_ret.kmx", occ, per, runs, pc, rh, maxbit + 1)
km.Circuit.from_file("/tmp/dqi_ret.kmx", format="kmx").to_file("/tmp/dqi_ret.kmb", format="kmb")
ret_bytes = pathlib.Path("/tmp/dqi_ret.kmb").read_bytes()
if sha(ret_bytes) == RET_KMB:
    (SUB / "dqi.kmb").write_bytes(ret_bytes); log("dqi with 2 retained products", sha(ret_bytes)[:12])
else:
    log("WARNING: retained build differs from the pinned file; keeping 4c81b726 unchanged", sha(ret_bytes)[:12])
for p in ("/tmp/dqi_base.kmx", "/tmp/dqi_ret.kmx"): os.remove(p)

## invsqrt: our splice of WCao24's normaliser and Thomas Gamet's early prefix
Two public improvements of the 62-qubit circuit sit in different places: WCao24's 44-CSWAP normaliser (`94b29cc5`)
and Thomas Gamet's early-prefix circuit (`5ec4f516`). We join public 62-qubit circuits at *clean cuts* (empty
condition stack, no classical bit written before the cut and read after it) where the wires and the phase agree on
every sampled input, and take the cheapest path (method and code in our analysis notebook). The segments below are
that path; the result is pinned by SHA-256 and verified by the official verifier further down.

In [ ]:
SEGMENTS = [["034c7c2a54d5", 0, 69], ["13b5bf7a18bc", 81, 83], ["26ebf39df3b1", 71, 92], ["94b29cc52689", 118, 479],
            ["034c7c2a54d5", 235, 421], ["5ec4f516164c", 515, 36611], ["034c7c2a54d5", 36245, 36409], ["5ec4f516164c", 36867, 36869],
            ["6c6a734dbcb6", 36857, 36859], ["94b29cc52689", 36549, 36553], ["13b5bf7a18bc", 53614, 53616], ["034c7c2a54d5", 36419, 36423],
            ["13b5bf7a18bc", 53620, 53622], ["26ebf39df3b1", 36634, 36636], ["ac540b8eff24", 46889, 46897], ["034c7c2a54d5", 36437, 36477]]
INV_SHA = "9c708f7b9abc7c7e6719f6e258a3881945e9ee1fd4fd7009fe37367ca1ef20d1"
D = pathlib.Path("/tmp/inv"); D.mkdir(exist_ok=True)
need = {s_[0] for s_ in SEGMENTS}; got = {}
for p in inp.rglob("*.kmb"):
    if "/invsqrt/" in str(p) and p.stem[:12] in need: got[p.stem[:12]] = p.read_bytes()
for b in (wcao["invsqrt.kmb"], thom["invsqrt.kmb"]): got[sha(b)[:12]] = b
for k, b in got.items():
    (D / f"{k}.kmb").write_bytes(b); km.Circuit.from_file(D / f"{k}.kmb", format="kmb").to_file(D / f"{k}.kmx", format="kmx")
def load_lines(name):
    head, body = [], []
    for line in open(D / f"{name}.kmx"):
        t = line.split()
        if not t: continue
        (head if t[0] in ("APPEND_TO_REGISTER", "REGISTER") else body).append(line.rstrip("\n"))
    return head, body
def remap_bits(line, table, counter):
    out = []
    for tok in line.split():
        if tok.startswith("b") and tok[1:].isdigit():
            if tok not in table: table[tok] = "b" + str(counter[0]); counter[0] += 1
            tok = table[tok]
        out.append(tok)
    return " ".join(out)

head, lines, counter = None, [], [0]
for name, a, b in SEGMENTS:
    h, body = load_lines(name); head = head or h; table = {}
    lines += [remap_bits(l, table, counter) for l in body[a:b]]
(D / "splice.kmx").write_text("\n".join(head + lines) + "\n")
km.Circuit.from_file(D / "splice.kmx", format="kmx").to_file(SUB / "invsqrt.kmb", format="kmb")
inv_sha = sha((SUB / "invsqrt.kmb").read_bytes()); log("invsqrt splice", inv_sha[:12])
assert inv_sha == INV_SHA, inv_sha

## Official verification of the exact bytes that go into `submission.zip`
The DQI is verified at the default salt and at `s1` and `s2` (the board uses its own draw; a decoder that fails on
none of these is the safest choice we can make).

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res, dqi_salts = {}, {}
KEYS = ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors", "repetition_factor")
for salt in ("", "s1", "s2"):
    r = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"] + (["--salt", salt] if salt else []))
    dqi_salts[salt or "default"] = {k: r.get(k) for k in KEYS}
    log("dqi salt", salt or "default", dqi_salts[salt or "default"])
    if not salt: res["dqi"] = r
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
dqi_zero = res["dqi"]["score"] / res["dqi"]["repetition_factor"] * 1.1912604648115832
for name, dq in (("this run's DQI draw", res["dqi"]["score"]), ("DQI at zero failed shots", dqi_zero)):
    r = {"invsqrt": REF["invsqrt"] / res["invsqrt"]["score"], "dqi": REF["dqi"] / dq, "dfthcsos": REF["dfthcsos"] / res["dfthcsos"]["score"]}
    print(f"{name:26s}", {k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 12, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res, "dqi_salts": dqi_salts},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP, "" if WRITE_SUBMISSION else "(set WRITE_SUBMISSION = True in your copy to save it in /kaggle/working)")

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
atlas = next((p for p in inp.rglob("circuits.csv") if "atlas" in str(p).lower()), None)
rows = {r["sha12"]: r for r in csv.DictReader(open(atlas))} if atlas else {}
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
**invsqrt** `9c708f7b`
- asteriospanoras (this notebook): the splice of the two circuits below at clean cuts, -3.0 Toffolis vs the best one.
- Thomas Gamet (with OpenAI Codex): the early-prefix circuit `5ec4f516`
  ([Quantum Attempt - new base bundle](https://www.kaggle.com/code/thomasgamet/quantum-attempt-new-base-bundle)).
- WCao24: the 44-CSWAP input normaliser `94b29cc5`, -2.0 Toffolis
  ([VQCC Deterministic Open Portfolio](https://www.kaggle.com/code/wcao24/vqcc-deterministic-open-portfolio)).
- maoxin123: `a15b0d73`, Block A shift-and-add stages re-ordered, controlled-SWAP rewrites
  ([LB 25.303 | Combined invsqrt + refit chem](https://www.kaggle.com/code/maoxin123/lb-25-303-combined-invsqrt-refit-chem)).
- manwithacat (James Barlow): HW-25 join and the public circuits dataset
  ([VQCC Hedge Wizardry](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)); Gengsr: the 46-CSWAP shifter
  ([LB 25.2268](https://www.kaggle.com/code/gengsr/lb-25-2268-shifter-46-own-chemistry-loaders)).
- The 62-qubit lineage: Syed Asad Ali (base circuit), Hosen42 (q25 CCX merge, measure-first rewrites), Kirill
  Tymchishin (carry port), Atakan Aldemir (CSWAP deletion), Ertuğrul Özer (`13b5bf7a` composition), busyaprime (Akmal
  Xodarev), yoyobar (19-bit conditional phase fusion), mrmeows / team selenium (the 63-qubit ancestor), Leo Tang and
  Ehan Mallick, as credited in their notebooks.

**DQI** `4c81b726` (maoxin123's 392k-Toffoli decoder, new in this notebook's portfolio)
- maoxin123: the decoder `4c81b726`, a gate-level rewrite of `cd71c482` in three waves (algebraic pull-backs of
  repeated bilinear and quadratic forms, clean-target gadgets in `unextract`, two waves of content-matched
  reductions), 392,016 mean Toffolis at 892 qubits, 0 failures at the default salt, `s1` and `s2`
  ([QCC v22 | a 392k-Toffoli DQI decoder](https://www.kaggle.com/code/maoxin123/qcc-v22-a-392k-toffoli-dqi-decoder),
  dataset [qcc-dqi-392k-toffoli-decoder](https://www.kaggle.com/datasets/maoxin123/qcc-dqi-392k-toffoli-decoder), CC0).
- Its base `cd71c482`: the team *i want to run gemma 9b*, Oleg Sher (sheroleg), Heliosli, Syed Asad Ali, SomeJoint and
  jday96314; the decoder line goes back to jday96314's continued-fraction decoder, SomeJoint's kit and narrow return,
  Ali Haydar Özdağ's flag carry and Syed Asad Ali's C1 line, with product reuse by Atakan Aldemir, MAIDANG/daifanhao,
  Isabe11aM00re and Heliosli, and Kirill Tymchishin's measurement identity (as credited by maoxin123).
- Areeb Ali: product retention (value fingerprints, idle windows, retention rewrite, exhaustive self-test); our
  planner extension adds 2 retained products on `4c81b726`
  ([Product Retention: 133 Free Toffolis](https://www.kaggle.com/code/areebbb/product-retention-133-free-toffolis-lb-25-40767)).
- The previous public decoders this one overtakes, credited for the ideas they made public: Areeb Ali's product
  retention `8eccde65` ([LB 25.40767](https://www.kaggle.com/code/areebbb/product-retention-133-free-toffolis-lb-25-40767)),
  Atakan Aldemir's iso-gap h5 `eae396ba` and our readout-window transplant (version 1 of our earlier notebook).

**Chemistry**
- maoxin123: v22 loaders, **4,089 Toffolis per pass** (five-block low-monomial subspace search), used from version 5
  ([QCC v22 | 4089-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4089-toffoli-chemistry-loaders)).
- Heliosli: weight update `e9cffb84` of prvsiyan's weights with prvsiyan's refit code, and the
  [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live) dataset this version reads
  ([The dataset is public now](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)).
- Lera (marculera): the (8, 143, 44) DF-THC table and its angle words
  ([dfthc-c44-build](https://www.kaggle.com/datasets/marculera/dfthc-c44-build)), and the authors her NOTICE credits
  (Ahmed Berat Özer, kozykappa, Kozykorpesh and Sumit Chongder, mrmeows, michaelrustad, phucthaiv02, somejoint,
  gengsr, qwertykeypadapple, lightningv08, syed3000).
- prvsiyan: constrained refit of the weights `f21b8a87` (used up to version 4)
  ([Verifier-Guided Quantum Circuit Optimization](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization)).
- maoxin123: v22 loaders, 4,097 Toffolis per pass (used up to version 4)
  ([QCC v22 | 4097-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4097-toffoli-chemistry-loaders)).
- ozertuu (Ertuğrul Özer): table `8afd4c67` and the cache dataset. Syed Asad Ali: the 8 x 143 x 44 table and the
  loader compiler. Lera (marculera): the DF-THC pipeline and the C44 build. Ahmed Berat Özer: progressive rounding and
  group compression. Atakan Aldemir: the pi-gauge. Weight refits before prvsiyan's: Isabe11aM00re, maoxin123 (with
  manwithacat's `dfthc_refit`), Heliosli, busyaprime. Dmitrii Gluzdov, milos and Agniv Banerjee: earlier shapes.

**Assembly and verification practice**: Alex Chilton's [Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
(pinning by SHA-256, re-verifying with the stock verifiers, pricing a DQI file at its board draw), and WCao24's
deterministic, byte-reproducible build. **Analysis**: Oleg Sher's [Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained).
Our companion analysis notebook: [QCC Toolkit: Simulator & Cost Anatomy](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).
Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; nothing here is
relicensed. The code of this notebook is Apache-2.0.